In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [3]:
# Load training and test data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [4]:
# Add a column to mark where the data came from
train["is_train"] = 1
test["is_train"] = 0
test["SalePrice"] = np.nan  # Add dummy target for test set


In [5]:
# Combine both datasets to preprocess together
combined = pd.concat([train, test], sort=False)

# Fill missing values
for column in combined.columns:
    if combined[column].dtype == "object":
        combined[column] = combined[column].fillna("None")
    else:
        combined[column] = combined[column].fillna(combined[column].median())

In [6]:
# Encode categorical features using LabelEncoder
label_columns = combined.select_dtypes(include="object").columns
for col in label_columns:
    combined[col] = LabelEncoder().fit_transform(combined[col])

# Feature Engineering
combined["TotalSF"] = combined["TotalBsmtSF"] + combined["1stFlrSF"] + combined["2ndFlrSF"]
combined["TotalBathrooms"] = (
    combined["FullBath"] + 0.5 * combined["HalfBath"] +
    combined["BsmtFullBath"] + 0.5 * combined["BsmtHalfBath"]
)
combined["Age"] = combined["YrSold"] - combined["YearBuilt"]
combined["RemodDiff"] = combined["YrSold"] - combined["YearRemodAdd"]
combined["WasRemodeled"] = (combined["YearBuilt"] != combined["YearRemodAdd"]).astype(int)


In [7]:
# Split back into train and test
train = combined[combined["is_train"] == 1].drop("is_train", axis=1)
test = combined[combined["is_train"] == 0].drop(["is_train", "SalePrice"], axis=1)

# Final training data
X = train.drop(["Id", "SalePrice"], axis=1)
y = train["SalePrice"]
X_test = test.drop("Id", axis=1)